In [1]:
import sys
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
# Add project root to Python path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

df = pd.read_csv(project_root / 'data' / 'processed' / 'model_training_data.csv')

In [2]:
target = 'race_is_winner'
group_col = 'event_id'

numeric_features = ['year','fp1_lap_count', 'fp1_lap_mean', 'fp1_lap_std', 'fp1_lap_best', 'fp1_lap_median', 'fp1_air_temp_mean', 'fp1_air_temp_std', 
                    'fp1_track_temp_mean', 'fp1_track_temp_std', 'fp1_humidity_mean', 'fp1_humidity_std', 'fp1_wind_speed_mean', 'fp1_wind_speed_std',
                    'fp1_rain_any', 'fp1_rain_samples_ratio', 'fp1_yellow_count', 'fp1_red_count', 'fp1_vsc_deployed_count', 'fp1_sc_deployed_count',
                    'fp1_disruption_ratio', 'fp1_best_free_practice_sec', 'fp1_free_practice_delta_to_best_lap_sec', 'fp1_free_practice_percent_of_best_lap_sec', 
                    'fp2_lap_count', 'fp2_lap_mean', 'fp2_lap_std', 'fp2_lap_best', 'fp2_lap_median', 'fp2_air_temp_mean', 'fp2_air_temp_std', 'fp2_track_temp_mean', 
                    'fp2_track_temp_std', 'fp2_humidity_mean', 'fp2_humidity_std', 'fp2_wind_speed_mean', 'fp2_wind_speed_std', 'fp2_rain_any', 'fp2_rain_samples_ratio', 
                    'fp2_yellow_count', 'fp2_red_count', 'fp2_vsc_deployed_count', 'fp2_sc_deployed_count', 'fp2_disruption_ratio', 'fp2_best_free_practice_sec', 
                    'fp2_free_practice_delta_to_best_lap_sec', 'fp2_free_practice_percent_of_best_lap_sec', 'fp3_lap_count', 'fp3_lap_mean', 'fp3_lap_std',
                    'fp3_lap_best', 'fp3_lap_median', 'fp3_air_temp_mean', 'fp3_air_temp_std', 'fp3_track_temp_mean', 'fp3_track_temp_std', 'fp3_humidity_mean', 
                    'fp3_humidity_std', 'fp3_wind_speed_mean', 'fp3_wind_speed_std', 'fp3_rain_any', 'fp3_rain_samples_ratio', 'fp3_yellow_count', 
                    'fp3_red_count', 'fp3_vsc_deployed_count', 'fp3_sc_deployed_count', 'fp3_disruption_ratio', 'fp3_best_free_practice_sec', 
                    'fp3_free_practice_delta_to_best_lap_sec', 'fp3_free_practice_percent_of_best_lap_sec', 'quali_lap_count', 'quali_lap_mean', 
                    'quali_lap_std', 'quali_lap_best', 'quali_lap_median', 'quali_air_temp_mean', 'quali_air_temp_std', 'quali_track_temp_mean', 
                    'quali_track_temp_std', 'quali_humidity_mean', 'quali_humidity_std', 'quali_wind_speed_mean', 'quali_wind_speed_std', 'quali_rain_any',
                    'quali_rain_samples_ratio', 'quali_yellow_count', 'quali_red_count', 'quali_vsc_deployed_count', 'quali_sc_deployed_count', 'quali_disruption_ratio', 
                    'quali_reached_q1', 'quali_reached_q2', 'quali_reached_q3', 'quali_best_quali_seconds', 'quali_quali_delta_to_pole', 'quali_quali_percent_of_pole', 'quali_quali_finish_position']

categorical_features = ['grand_prix', 'event_id', 'row_id', 'driver_id', 'abbreviation', 'team_name', 'team_id']

features = numeric_features + categorical_features 

X_train = df[features]
y_train = df[target]

In [3]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

model_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        class_weight="balanced",
    ))
])

model_pipeline.fit(X_train, y_train)

artifact = {
    'model': model_pipeline,
    'feature_cols': features,
    'target': target,
    'group_col': group_col,
}

# Next cell uses this dict in-memory via WinnerPredictor(artifact=artifact) — no joblib file needed.

In [4]:
from f1_prediction_ml.pipelines.build_inference_features import build_inference_features
from f1_prediction_ml.modeling.predict_winner import WinnerPredictor

# Uses the pipeline trained in the previous cell (same kernel session) — not models/*.pkl on disk.
predictor_a = WinnerPredictor(artifact=artifact)


def predict_model(predictor, year: int, grand_prix: str):
    print(type(year), type(grand_prix))
    next_race_df = build_inference_features(
        year=year,
        grand_prix=grand_prix,
        
    )

    predictions = predictor.predict_next_race_winner(next_race_df)
    winner = predictor.get_predicted_winner(predictions)

    return next_race_df, predictions, winner

In [5]:
next_race_df_a, predictions_a, winner_a = predict_model(predictor_a, 2026, "China")

core           INFO 	Loading data for Chinese Grand Prix - Practice 1 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


<class 'int'> <class 'str'>
INFO: Detected EventFormat = 'sprint_qualifying'
INFO: Sprint weekend detected (sprint_qualifying) — using ['FP1', 'SQ', 'S', 'SS', 'Q']


req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Chinese Grand Prix - Sprint Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2026 China FP1 session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11235, 'Type': 'Practice', 'Number': 1, 'Name': 'Practice 1', 'StartDate': datetime.datetime(2026, 3, 13, 11, 30), 'EndDate': datetime.datetime(2026, 3, 13, 12, 30), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-13_Practice_1/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:21.976000      11.8      40.8    1030.3     False        11.2   
1 0 days 00:01:21.984000      11.8      40.8    1030.3  

core        WARNING 	No lap data for driver 11
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 11)
req            INFO 	Using cached data for weather_data
core        WARNING 	Cannot calculate qualifying results: missing information about deleted laps. Make sure that race control messages are being loaded.
logger      WARNING 	Failed to calculate quali results from lap times!
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '1', '81', '16', '44', '87', '3', '27', '10', '30', '5', '6', '31', '43', '23', '55', '14', '77', '18', '41', '11']
core           INFO 	Loading data for Chinese Grand Prix - Sprint [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 

INFO: Fetched data for 2026 China SQ session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11236, 'Type': 'Qualifying', 'Name': 'Sprint Qualifying', 'StartDate': datetime.datetime(2026, 3, 13, 15, 30), 'EndDate': datetime.datetime(2026, 3, 13, 16, 14), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-13_Sprint_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:59.719000      16.3      23.9    1026.7     False        30.6   
1 0 days 00:01:59.702000      16.3      23.0    1026.7

req            INFO 	Using cached data for weather_data
core        WARNING 	Driver 63 completed the race distance 00:00.009000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['63', '16', '44', '1', '12', '81', '30', '87', '3', '31', '10', '55', '5', '43', '6', '23', '14', '18', '11', '27', '77', '41']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2026 China S session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11240, 'Type': 'Race', 'Name': 'Sprint', 'StartDate': datetime.datetime(2026, 3, 14, 11, 0), 'EndDate': datetime.datetime(2026, 3, 14, 12, 0), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-14_Sprint/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:13.409000      13.9      54.3    1027.0     False        12.2   
1 0 days 00:01:13.437000      13.9      54.2    1027.0     False        12.4   
2 0 d

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '81', '1', '10', '3', '6', '87', '27', '43', '31', '30', '41', '5', '55', '23', '14', '77', '18', '11']


INFO: Fetched data for 2026 China Q session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11241, 'Type': 'Qualifying', 'Name': 'Qualifying', 'StartDate': datetime.datetime(2026, 3, 14, 15, 0), 'EndDate': datetime.datetime(2026, 3, 14, 16, 0), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-14_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:18.401000      17.6      36.5    1023.7     False        32.3   
1 0 days 00:01:18.410000      17.6      36.2    1023.6     False       

## <font color='teal'>The model predicted the winner correctly. The driver is new and the model had no information about him in the training data. It is likely that the model learned to use qualifying results to predict the winner. Fastest person in qualifying is most likely to win. Win probabilty is also very low, yet the model picked the driver.</font>

In [6]:
numeric_features_reduced = ['year','fp1_lap_count', 'fp1_lap_mean', 'fp1_lap_std', 'fp1_lap_best', 'fp1_lap_median', 'fp1_air_temp_mean', 'fp1_air_temp_std', 
                    'fp1_track_temp_mean', 'fp1_track_temp_std', 'fp1_humidity_mean', 'fp1_humidity_std', 'fp1_wind_speed_mean', 'fp1_wind_speed_std',
                    'fp1_rain_any', 'fp1_rain_samples_ratio', 'fp1_yellow_count', 'fp1_red_count', 'fp1_vsc_deployed_count', 'fp1_sc_deployed_count',
                    'fp1_disruption_ratio', 'fp1_best_free_practice_sec', 'fp1_free_practice_delta_to_best_lap_sec', 'fp1_free_practice_percent_of_best_lap_sec', 
                    'fp2_lap_count', 'fp2_lap_mean', 'fp2_lap_std', 'fp2_lap_best', 'fp2_lap_median', 'fp2_air_temp_mean', 'fp2_air_temp_std', 'fp2_track_temp_mean', 
                    'fp2_track_temp_std', 'fp2_humidity_mean', 'fp2_humidity_std', 'fp2_wind_speed_mean', 'fp2_wind_speed_std', 'fp2_rain_any', 'fp2_rain_samples_ratio', 
                    'fp2_yellow_count', 'fp2_red_count', 'fp2_vsc_deployed_count', 'fp2_sc_deployed_count', 'fp2_disruption_ratio', 'fp2_best_free_practice_sec', 
                    'fp2_free_practice_delta_to_best_lap_sec', 'fp2_free_practice_percent_of_best_lap_sec', 'fp3_lap_count', 'fp3_lap_mean', 'fp3_lap_std',
                    'fp3_lap_best', 'fp3_lap_median', 'fp3_air_temp_mean', 'fp3_air_temp_std', 'fp3_track_temp_mean', 'fp3_track_temp_std', 'fp3_humidity_mean', 
                    'fp3_humidity_std', 'fp3_wind_speed_mean', 'fp3_wind_speed_std', 'fp3_rain_any', 'fp3_rain_samples_ratio', 'fp3_yellow_count', 
                    'fp3_red_count', 'fp3_vsc_deployed_count', 'fp3_sc_deployed_count', 'fp3_disruption_ratio', 'fp3_best_free_practice_sec', 
                    'fp3_free_practice_delta_to_best_lap_sec', 'fp3_free_practice_percent_of_best_lap_sec', 'quali_lap_count', 'quali_lap_mean', 
                    'quali_lap_std', 'quali_lap_best', 'quali_lap_median', 'quali_air_temp_mean', 'quali_air_temp_std', 'quali_track_temp_mean', 
                    'quali_track_temp_std', 'quali_humidity_mean', 'quali_humidity_std', 'quali_wind_speed_mean', 'quali_wind_speed_std', 'quali_rain_any',
                    'quali_rain_samples_ratio', 'quali_yellow_count', 'quali_red_count', 'quali_vsc_deployed_count', 'quali_sc_deployed_count', 'quali_disruption_ratio', 
                    'quali_reached_q1', 'quali_reached_q2', 'quali_reached_q3']

categorical_features = ['grand_prix', 'event_id', 'row_id', 'driver_id', 'abbreviation', 'team_name', 'team_id']

reduced_features = numeric_features_reduced + categorical_features 

X_train_reduced = df[reduced_features]
y_train = df[target]

In [7]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scalar', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features_reduced),
    ('cat', categorical_transformer, categorical_features),
], remainder='drop')

model2_pipeline = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=42,
        class_weight="balanced",
    ))
])

model2_pipeline.fit(X_train_reduced, y_train)

artifact2 = {
    'model': model2_pipeline,
    'feature_cols': reduced_features,
    'target': target,
    'group_col': group_col,
}

# Next cell uses this dict in-memory via WinnerPredictor(artifact=artifact) — no joblib file needed.

In [8]:
# Uses the pipeline trained in the previous cell (same kernel session) — not models/*.pkl on disk.
predictor_b = WinnerPredictor(artifact=artifact2)

next_race_df_b, predictions_b, winner_b = predict_model(predictor_b, 2026, "China")

core           INFO 	Loading data for Chinese Grand Prix - Practice 1 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


<class 'int'> <class 'str'>
INFO: Detected EventFormat = 'sprint_qualifying'
INFO: Sprint weekend detected (sprint_qualifying) — using ['FP1', 'SQ', 'S', 'SS', 'Q']


req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['1', '3', '5', '6', '10', '11', '12', '14', '16', '18', '23', '27', '30', '31', '41', '43', '44', '55', '63', '77', '81', '87']
core           INFO 	Loading data for Chinese Grand Prix - Sprint Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core        WARNING 	Sprint Qualifying is not supported by Ergast! Limited results are calculated from timing data.
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2026 China FP1 session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11235, 'Type': 'Practice', 'Number': 1, 'Name': 'Practice 1', 'StartDate': datetime.datetime(2026, 3, 13, 11, 30), 'EndDate': datetime.datetime(2026, 3, 13, 12, 30), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-13_Practice_1/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:21.976000      11.8      40.8    1030.3     False        11.2   
1 0 days 00:01:21.984000      11.8      40.8    1030.3  

core        WARNING 	No lap data for driver 11
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 11)
req            INFO 	Using cached data for weather_data
core        WARNING 	Cannot calculate qualifying results: missing information about deleted laps. Make sure that race control messages are being loaded.
logger      WARNING 	Failed to calculate quali results from lap times!
core           INFO 	Finished loading data for 22 drivers: ['63', '12', '1', '81', '16', '44', '87', '3', '27', '10', '30', '5', '6', '31', '43', '23', '55', '14', '77', '18', '41', '11']
core           INFO 	Loading data for Chinese Grand Prix - Sprint [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 

INFO: Fetched data for 2026 China SQ session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11236, 'Type': 'Qualifying', 'Name': 'Sprint Qualifying', 'StartDate': datetime.datetime(2026, 3, 13, 15, 30), 'EndDate': datetime.datetime(2026, 3, 13, 16, 14), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-13_Sprint_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:59.719000      16.3      23.9    1026.7     False        30.6   
1 0 days 00:01:59.702000      16.3      23.0    1026.7

req            INFO 	Using cached data for weather_data
core        WARNING 	Driver 63 completed the race distance 00:00.009000 before the recorded end of the session.
core           INFO 	Finished loading data for 22 drivers: ['63', '16', '44', '1', '12', '81', '30', '87', '3', '31', '10', '55', '5', '43', '6', '23', '14', '18', '11', '27', '77', '41']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2026 China S session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11240, 'Type': 'Race', 'Name': 'Sprint', 'StartDate': datetime.datetime(2026, 3, 14, 11, 0), 'EndDate': datetime.datetime(2026, 3, 14, 12, 0), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-14_Sprint/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:13.409000      13.9      54.3    1027.0     False        12.2   
1 0 days 00:01:13.437000      13.9      54.2    1027.0     False        12.4   
2 0 d

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 22 drivers: ['12', '63', '44', '16', '81', '1', '10', '3', '6', '87', '27', '43', '31', '30', '41', '5', '55', '23', '14', '77', '18', '11']


INFO: Fetched data for 2026 China Q session.
********** Session Info **********
{'Meeting': {'Key': 1280, 'Name': 'Chinese Grand Prix', 'OfficialName': 'FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026', 'Location': 'Shanghai', 'Number': 2, 'Country': {'Key': 53, 'Code': 'CHN', 'Name': 'China'}, 'Circuit': {'Key': 49, 'ShortName': 'Shanghai'}}, 'SessionStatus': 'Inactive', 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 11241, 'Type': 'Qualifying', 'Name': 'Qualifying', 'StartDate': datetime.datetime(2026, 3, 14, 15, 0), 'EndDate': datetime.datetime(2026, 3, 14, 16, 0), 'GmtOffset': datetime.timedelta(seconds=28800), 'Path': '2026/2026-03-15_Chinese_Grand_Prix/2026-03-14_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:18.401000      17.6      36.5    1023.7     False        32.3   
1 0 days 00:01:18.410000      17.6      36.2    1023.6     False       

In [9]:
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss, roc_auc_score

def evaluate_winner_model(
    df,
    model,
    feature_cols,
    target="race_is_winner",
    group_col="event_id",
    n_splits=3
):
    data = df.copy()
    data = data.dropna(subset=[target])

    missing_features = [col for col in feature_cols if col not in data.columns]
    if missing_features:
        raise ValueError(f"Missing features: {missing_features}")

    X = data[feature_cols].copy()
    y = data[target].astype(int)
    groups = data[group_col]

    gkf = GroupKFold(n_splits=n_splits)

    winner_pick_scores = []
    top3_hit_scores = []
    roc_auc_scores = []
    log_loss_scores = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        test_groups = groups.iloc[test_idx]

        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]

        fold_eval = pd.DataFrame({
            "event_id": test_groups.values,
            "y_true": y_test.values,
            "y_proba": y_proba
        })

        # winner pick accuracy
        predicted_winners = (
            fold_eval.sort_values(["event_id", "y_proba"], ascending=[True, False])
                     .groupby("event_id", as_index=False)
                     .head(1)
        )
        winner_pick_acc = predicted_winners["y_true"].mean()
        winner_pick_scores.append(winner_pick_acc)

        # top-3 hit rate
        top3 = (
            fold_eval.sort_values(["event_id", "y_proba"], ascending=[True, False])
                     .groupby("event_id", as_index=False)
                     .head(3)
        )
        top3_hit = top3.groupby("event_id")["y_true"].max().mean()
        top3_hit_scores.append(top3_hit)

        # secondary metrics
        try:
            roc_auc_scores.append(roc_auc_score(y_test, y_proba))
        except ValueError:
            pass

        try:
            log_loss_scores.append(log_loss(y_test, y_proba, labels=[0, 1]))
        except ValueError:
            pass

        print(
            f"Fold {fold}: "
            f"winner_pick_accuracy={winner_pick_acc:.4f}, "
            f"top3_hit_rate={top3_hit:.4f}"
        )

    results = {
        "mean_winner_pick_accuracy": sum(winner_pick_scores) / len(winner_pick_scores),
        "mean_top3_hit_rate": sum(top3_hit_scores) / len(top3_hit_scores),
        "mean_roc_auc": sum(roc_auc_scores) / len(roc_auc_scores) if roc_auc_scores else None,
        "mean_log_loss": sum(log_loss_scores) / len(log_loss_scores) if log_loss_scores else None,
    }

    print("\nSummary:")
    for k, v in results.items():
        if v is not None:
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: None")

    return results

In [10]:
print("=== Model A: with strong quali features ===")
results_a = evaluate_winner_model(df, model_pipeline, features)

print("\n=== Model B: reduced quali dependence ===")
results_b = evaluate_winner_model(df, model2_pipeline, reduced_features)

print("Model A winner pick:")
print(winner_a[["event_id", "abbreviation", "win_proba"]])

print("\nModel B winner pick:")
print(winner_b[["event_id", "abbreviation", "win_proba"]])

=== Model A: with strong quali features ===
Fold 1: winner_pick_accuracy=0.5000, top3_hit_rate=0.8636
Fold 2: winner_pick_accuracy=0.4286, top3_hit_rate=0.7619
Fold 3: winner_pick_accuracy=0.6190, top3_hit_rate=0.9048

Summary:
mean_winner_pick_accuracy: 0.5159
mean_top3_hit_rate: 0.8434
mean_roc_auc: 0.9411
mean_log_loss: 0.1227

=== Model B: reduced quali dependence ===
Fold 1: winner_pick_accuracy=0.4545, top3_hit_rate=0.8636
Fold 2: winner_pick_accuracy=0.4286, top3_hit_rate=0.7143
Fold 3: winner_pick_accuracy=0.4762, top3_hit_rate=0.8095

Summary:
mean_winner_pick_accuracy: 0.4531
mean_top3_hit_rate: 0.7958
mean_roc_auc: 0.9190
mean_log_loss: 0.1341
Model A winner pick:
                  event_id abbreviation  win_proba
0  2026_Chinese_Grand_Prix          ANT   0.133333

Model B winner pick:
                  event_id abbreviation  win_proba
0  2026_Chinese_Grand_Prix          VER   0.153333


In [11]:
print("=== Model A top 10 ===")
print(predictions_a.head(10))

print("\n=== Model B top 10 ===")
print(predictions_b.head(10))

=== Model A top 10 ===
                  event_id       driver_id abbreviation  win_proba
0  2026_Chinese_Grand_Prix       antonelli          ANT   0.133333
1  2026_Chinese_Grand_Prix         russell          RUS   0.123333
2  2026_Chinese_Grand_Prix  max_verstappen          VER   0.120000
3  2026_Chinese_Grand_Prix        hamilton          HAM   0.113333
4  2026_Chinese_Grand_Prix         piastri          PIA   0.100000
5  2026_Chinese_Grand_Prix          hadjar          HAD   0.096667
6  2026_Chinese_Grand_Prix         leclerc          LEC   0.090000
7  2026_Chinese_Grand_Prix          norris          NOR   0.086667
8  2026_Chinese_Grand_Prix           gasly          GAS   0.073333
9  2026_Chinese_Grand_Prix          lawson          LAW   0.066667

=== Model B top 10 ===
                  event_id       driver_id abbreviation  win_proba
0  2026_Chinese_Grand_Prix  max_verstappen          VER   0.153333
1  2026_Chinese_Grand_Prix          hadjar          HAD   0.123333
2  2026_Chinese

## <font color='teal'>The qualifying-related features are genuinely adding predictive value. Removing them made the model less accurate, less sharp, and slightly worse at ranking likely winners. Model A is the stronger post-qualifying predictor, but part of that strength comes from heavy reliance on qualifying performance.</font>
In the next section both models will be tested on a race where the qinner of the quali session did not win the race and finished 7th.

In [12]:
next_race_df_a, predictions_a, winner_a = predict_model(predictor_a, 2020, "Monza")
next_race_df_b, predictions_b, winner_b = predict_model(predictor_b, 2020, "Monza")

<class 'int'> <class 'str'>
INFO: Detected EventFormat = 'conventional'

core           INFO 	Loading data for Italian Grand Prix - Practice 1 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '7', '8', '10', '11', '16', '18', '20', '23', '26', '31', '33', '40', '44', '55', '77', '99']
core           INFO 	Loading data for Italian Grand Prix - Practice 2 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '7'
core        WARNING 	Fixed incorrect tyre stint information for driver '8'
core        WARNING 	Fixed incorrect tyre stint information for driver '20'


INFO: Fetched data for 2020 Monza FP1 session.
********** Session Info **********
{'Meeting': {'Key': 1052, 'Name': 'Italian Grand Prix', 'Location': 'Monza', 'Country': {'Key': 13, 'Code': 'ITA', 'Name': 'Italy'}, 'Circuit': {'Key': 39, 'ShortName': 'Monza'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 5801, 'Type': 'Practice', 'Number': 1, 'Name': 'Practice 1', 'StartDate': datetime.datetime(2020, 9, 4, 11, 0), 'EndDate': datetime.datetime(2020, 9, 4, 12, 30), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2020/2020-09-06_Italian_Grand_Prix/2020-09-04_Practice_1/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:23.321000      23.3      58.1    1006.0     False        30.4   
1 0 days 00:01:23.322000      23.3      57.4    1006.0     False        30.7   
2 0 days 00:02:23.327000      23.3      57.8    1006.0     False        31.2   
3 0 days 00

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '7', '8', '10', '11', '16', '18', '20', '23', '26', '31', '33', '44', '55', '63', '77', '99']
core           INFO 	Loading data for Italian Grand Prix - Practice 3 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

INFO: Fetched data for 2020 Monza FP2 session.
********** Session Info **********
{'Meeting': {'Key': 1052, 'Name': 'Italian Grand Prix', 'Location': 'Monza', 'Country': {'Key': 13, 'Code': 'ITA', 'Name': 'Italy'}, 'Circuit': {'Key': 39, 'ShortName': 'Monza'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 5802, 'Type': 'Practice', 'Number': 2, 'Name': 'Practice 2', 'StartDate': datetime.datetime(2020, 9, 4, 15, 0), 'EndDate': datetime.datetime(2020, 9, 4, 16, 30), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2020/2020-09-06_Italian_Grand_Prix/2020-09-04_Practice_2/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:55.955000      26.9      46.1    1004.3     False        43.6   
1 0 days 00:01:55.948000      26.7      46.3    1004.3     False        43.6   
2 0 days 00:02:55.956000      26.5      47.0    1004.3     False        43.7   
3 0 days 00

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '7', '8', '10', '11', '16', '18', '20', '23', '26', '31', '33', '44', '55', '63', '77', '99']
core           INFO 	Loading data for Italian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2020 Monza FP3 session.
********** Session Info **********
{'Meeting': {'Key': 1052, 'Name': 'Italian Grand Prix', 'Location': 'Monza', 'Country': {'Key': 13, 'Code': 'ITA', 'Name': 'Italy'}, 'Circuit': {'Key': 39, 'ShortName': 'Monza'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 5803, 'Type': 'Practice', 'Number': 3, 'Name': 'Practice 3', 'StartDate': datetime.datetime(2020, 9, 5, 12, 0), 'EndDate': datetime.datetime(2020, 9, 5, 13, 0), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2020/2020-09-06_Italian_Grand_Prix/2020-09-05_Practice_3/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:28.617000      26.0      54.0     998.4     False        36.0   
1 0 days 00:01:28.618000      25.7      54.7     998.4     False        36.3   
2 0 days 00:02:28.625000      25.8      54.9     998.4     False        36.3   
3 0 days 00:

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '55', '11', '33', '4', '3', '18', '23', '10', '26', '31', '16', '7', '20', '8', '5', '99', '63', '6']


INFO: Fetched data for 2020 Monza Q session.
********** Session Info **********
{'Meeting': {'Key': 1052, 'Name': 'Italian Grand Prix', 'Location': 'Monza', 'Country': {'Key': 13, 'Code': 'ITA', 'Name': 'Italy'}, 'Circuit': {'Key': 39, 'ShortName': 'Monza'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 5804, 'Type': 'Qualifying', 'Name': 'Qualifying', 'StartDate': datetime.datetime(2020, 9, 5, 15, 0), 'EndDate': datetime.datetime(2020, 9, 5, 16, 0), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2020/2020-09-06_Italian_Grand_Prix/2020-09-05_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:28.436000      28.0      46.7     996.2     False        44.8   
1 0 days 00:01:28.444000      28.2      46.5     996.2     False        44.9   
2 0 days 00:02:28.440000      28.3      46.0     996.3     False        45.1   
3 0 days 00:03:28.440000 

In [13]:
print("Model A winner pick:")
print(winner_a[["event_id", "abbreviation", "win_proba"]])

print("\nModel B winner pick:")
print(winner_b[["event_id", "abbreviation", "win_proba"]])

Model A winner pick:
                  event_id abbreviation  win_proba
0  2020_Italian_Grand_Prix          HAM   0.533333

Model B winner pick:
                  event_id abbreviation  win_proba
0  2020_Italian_Grand_Prix          HAM   0.433333


In [14]:
print("=== Model A top 10 ===")
print(predictions_a.head(10))

print("\n=== Model B top 10 ===")
print(predictions_b.head(10))

=== Model A top 10 ===
                  event_id       driver_id abbreviation  win_proba
0  2020_Italian_Grand_Prix        hamilton          HAM   0.533333
1  2020_Italian_Grand_Prix          bottas          BOT   0.336667
2  2020_Italian_Grand_Prix  max_verstappen          VER   0.180000
3  2020_Italian_Grand_Prix           sainz          SAI   0.070000
4  2020_Italian_Grand_Prix       ricciardo          RIC   0.050000
5  2020_Italian_Grand_Prix           albon          ALB   0.043333
6  2020_Italian_Grand_Prix           perez          PER   0.026667
7  2020_Italian_Grand_Prix          norris          NOR   0.020000
8  2020_Italian_Grand_Prix          vettel          VET   0.016667
9  2020_Italian_Grand_Prix          stroll          STR   0.013333

=== Model B top 10 ===
                  event_id       driver_id abbreviation  win_proba
0  2020_Italian_Grand_Prix        hamilton          HAM   0.433333
1  2020_Italian_Grand_Prix          bottas          BOT   0.373333
2  2020_Italian

In [15]:
next_race_df_a, predictions_a, winner_a = predict_model(predictor_a, 2021, "Hungary")
next_race_df_b, predictions_b, winner_b = predict_model(predictor_b, 2021, "Hungary")

<class 'int'> <class 'str'>


core           INFO 	Loading data for Hungarian Grand Prix - Practice 1 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


INFO: Detected EventFormat = 'conventional'


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '88'
core        WARNING 	Driver 88: Lap timing integrity check failed for 1 lap(s)
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '9', '10', '11', '14', '16', '18', '22', '31', '33', '44', '47', '55', '63', '77', '88', '99']
core           INFO 	Loading data for Hungarian Grand Prix - Practice 2 [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req

INFO: Fetched data for 2021 Hungary FP1 session.
********** Session Info **********
{'Meeting': {'Key': 1073, 'Name': 'Hungarian Grand Prix', 'Location': 'Budapest', 'Country': {'Key': 14, 'Code': 'HUN', 'Name': 'Hungary'}, 'Circuit': {'Key': 4, 'ShortName': 'Hungaroring'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 6255, 'Type': 'Practice', 'Number': 1, 'Name': 'Practice 1', 'StartDate': datetime.datetime(2021, 7, 30, 11, 30), 'EndDate': datetime.datetime(2021, 7, 30, 12, 30), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2021/2021-08-01_Hungarian_Grand_Prix/2021-07-30_Practice_1/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:21.289000      30.0      39.0     985.6     False        50.8   
1 0 days 00:01:21.287000      30.0      38.4     985.7     False        51.2   
2 0 days 00:02:21.294000      30.1      38.4     985.7     False        

...

INFO: Fetched data for 2021 Hungary FP2 session.
********** Session Info **********
{'Meeting': {'Key': 1073, 'Name': 'Hungarian Grand Prix', 'Location': 'Budapest', 'Country': {'Key': 14, 'Code': 'HUN', 'Name': 'Hungary'}, 'Circuit': {'Key': 4, 'ShortName': 'Hungaroring'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 6256, 'Type': 'Practice', 'Number': 2, 'Name': 'Practice 2', 'StartDate': datetime.datetime(2021, 7, 30, 15, 0), 'EndDate': datetime.datetime(2021, 7, 30, 16, 0), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2021/2021-08-01_Hungarian_Grand_Prix/2021-07-30_Practice_2/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:21.674000      32.0      34.4     984.0     False        64.1   
1 0 days 00:01:21.671000      32.1      35.0     984.1     False        64.0   
2 0 days 00:02:21.671000      32.1      36.0     984.1     False        63

req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['3', '4', '5', '6', '7', '9', '10', '11', '14', '16', '18', '22', '31', '33', '44', '47', '55', '63', '77', '99']
core           INFO 	Loading data for Hungarian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


INFO: Fetched data for 2021 Hungary FP3 session.
********** Session Info **********
{'Meeting': {'Key': 1073, 'Name': 'Hungarian Grand Prix', 'Location': 'Budapest', 'Country': {'Key': 14, 'Code': 'HUN', 'Name': 'Hungary'}, 'Circuit': {'Key': 4, 'ShortName': 'Hungaroring'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 6257, 'Type': 'Practice', 'Number': 3, 'Name': 'Practice 3', 'StartDate': datetime.datetime(2021, 7, 31, 12, 0), 'EndDate': datetime.datetime(2021, 7, 31, 13, 0), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2021/2021-08-01_Hungarian_Grand_Prix/2021-07-31_Practice_3/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:25.697000      26.6      59.5     982.3     False        49.3   
1 0 days 00:01:25.706000      26.8      59.3     982.3     False        49.4   
2 0 days 00:02:25.720000      26.8      58.9     982.3     False        49

core        WARNING 	No lap data for driver 47
core        WARNING 	Failed to perform lap accuracy check - all laps marked as inaccurate (driver 47)
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '33', '11', '10', '4', '16', '31', '14', '5', '3', '18', '7', '99', '55', '22', '63', '6', '9', '47']


INFO: Fetched data for 2021 Hungary Q session.
********** Session Info **********
{'Meeting': {'Key': 1073, 'Name': 'Hungarian Grand Prix', 'Location': 'Budapest', 'Country': {'Key': 14, 'Code': 'HUN', 'Name': 'Hungary'}, 'Circuit': {'Key': 4, 'ShortName': 'Hungaroring'}}, 'ArchiveStatus': {'Status': 'Generating'}, 'Key': 6258, 'Type': 'Qualifying', 'Name': 'Qualifying', 'StartDate': datetime.datetime(2021, 7, 31, 15, 0), 'EndDate': datetime.datetime(2021, 7, 31, 16, 0), 'GmtOffset': datetime.timedelta(seconds=7200), 'Path': '2021/2021-08-01_Hungarian_Grand_Prix/2021-07-31_Qualifying/'}
INFO: Aggregating weather data
************ Weather DataFrame Head ***********
                    time  air_temp  humidity  pressure  rainfall  track_temp  \
0 0 days 00:00:24.530000      29.1      45.4     982.2     False        59.7   
1 0 days 00:01:24.530000      29.1      45.4     982.3     False        59.4   
2 0 days 00:02:24.533000      29.0      44.8     982.3     False        59.4   
3 0 day

In [16]:
print("Model A winner pick:")
print(winner_a[["event_id", "abbreviation", "win_proba"]])

print("\nModel B winner pick:")
print(winner_b[["event_id", "abbreviation", "win_proba"]])

Model A winner pick:
                    event_id abbreviation  win_proba
0  2021_Hungarian_Grand_Prix          VER   0.286667

Model B winner pick:
                    event_id abbreviation  win_proba
0  2021_Hungarian_Grand_Prix          VER   0.203333


In [17]:
print("=== Model A top 10 ===")
print(predictions_a.head(10))

print("\n=== Model B top 10 ===")
print(predictions_b.head(10))

=== Model A top 10 ===
                    event_id       driver_id abbreviation  win_proba
0  2021_Hungarian_Grand_Prix  max_verstappen          VER   0.286667
1  2021_Hungarian_Grand_Prix        hamilton          HAM   0.263333
2  2021_Hungarian_Grand_Prix          bottas          BOT   0.163333
3  2021_Hungarian_Grand_Prix           perez          PER   0.096667
4  2021_Hungarian_Grand_Prix         leclerc          LEC   0.046667
5  2021_Hungarian_Grand_Prix            ocon          OCO   0.040000
6  2021_Hungarian_Grand_Prix           gasly          GAS   0.033333
7  2021_Hungarian_Grand_Prix           sainz          SAI   0.026667
8  2021_Hungarian_Grand_Prix      giovinazzi          GIO   0.023333
9  2021_Hungarian_Grand_Prix          alonso          ALO   0.020000

=== Model B top 10 ===
                    event_id       driver_id abbreviation  win_proba
0  2021_Hungarian_Grand_Prix  max_verstappen          VER   0.203333
1  2021_Hungarian_Grand_Prix        hamilton          HA

## <font color="teal">The full post-qualifying model performed best overall, but stress-testing on Monza 2020 and Hungary 2021 showed that both model variants still favored dominant pre-race contenders over rare upset outcomes. Removing the strongest qualifying features reduced the model’s dependence on qualifying position, but the reduced-feature model still struggled to identify low-probability chaos winners like Pierre Gasly at Monza 2020 and Esteban Ocon at Hungary 2021.</font>